In [0]:
WITH active_members AS (
    SELECT cde_id, plan_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = (SELECT MAX(snapshot_date_id)
                              FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot)
      AND renewal_status LIKE 'Active%'
),
indian_film_viewers AS (
    SELECT t.CDE_ID
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_film f
      ON t.film_id = f.film_id
    WHERE t.TicketFLAG = 'Yes'
      AND f.language IN ('Hindi','Punjabi','Gujarati','Telugu','Tamil')
      AND t.VisitDateId >= CAST(date_format(add_months(current_date(), -12), 'yyyyMMdd') AS INT)
      AND t.VisitDateId <  CAST(date_format(current_date(), 'yyyyMMdd') AS INT)
      AND t.CDE_ID > 0
    GROUP BY t.CDE_ID
    HAVING SUM(t.Quantity) > 0          -- refunds netted out
)
SELECT
    a.plan_id,
    COUNT(DISTINCT a.cde_id)                                   AS active_members,
    COUNT(DISTINCT v.CDE_ID)                                   AS watched_indian_film_p12m,
    ROUND(COUNT(DISTINCT v.CDE_ID) / COUNT(DISTINCT a.cde_id) * 100, 1) AS pct
FROM active_members a
LEFT JOIN indian_film_viewers v
  ON a.cde_id = v.CDE_ID
GROUP BY a.plan_id
ORDER BY a.plan_id;
